In [586]:
# Updated Cell 1: Imports

import os
import numpy as np
import pandas as pd
import scipy.io as sio
import pywt
# Updated Cell 1: Add XGBoost Import

from xgboost import XGBClassifier
# Updated Cell 1: Add This Import

from sklearn.utils.class_weight import compute_sample_weight

from scipy.stats import kurtosis, skew
from scipy.signal import welch, hilbert
from scipy.fft import rfft, rfftfreq

from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [587]:
# Updated Cell 2: Feature Extraction Function

def extract_signal_features(signal, fs, rpm=None, placement_name='Unknown'):
    signal = np.array(signal).flatten()

    if len(signal) == 0:
        return None

    mean_val = np.mean(signal)
    std_val = np.std(signal)
    rms = np.sqrt(np.mean(signal ** 2))
    peak = np.max(np.abs(signal))
    peak_to_peak = np.ptp(signal)
    crest_factor = peak / (rms + 1e-8)
    kurt = kurtosis(signal)
    sk = skew(signal)

    freqs, psd = welch(signal, fs=fs, nperseg=min(1024, len(signal)))

    dominant_freq = freqs[np.argmax(psd)] if len(freqs) > 0 else 0
    max_psd = np.max(psd) if len(psd) > 0 else 0
    mean_psd = np.mean(psd) if len(psd) > 0 else 0

    top_indices = np.argsort(psd)[-3:]

    top_freq_1 = freqs[top_indices[-1]] if len(top_indices) >= 1 else 0
    top_freq_2 = freqs[top_indices[-2]] if len(top_indices) >= 2 else 0
    top_freq_3 = freqs[top_indices[-3]] if len(top_indices) >= 3 else 0

    top_psd_1 = psd[top_indices[-1]] if len(top_indices) >= 1 else 0
    top_psd_2 = psd[top_indices[-2]] if len(top_indices) >= 2 else 0
    top_psd_3 = psd[top_indices[-3]] if len(top_indices) >= 3 else 0

    # Spectral entropy
    psd_norm = psd / (np.sum(psd) + 1e-8)
    spectral_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-8))

    # Frequency band energy
    low_band_energy = np.sum(psd[(freqs >= 0) & (freqs < 100)])
    mid_band_energy = np.sum(psd[(freqs >= 100) & (freqs < 500)])
    high_band_energy = np.sum(psd[(freqs >= 500) & (freqs < 2000)])

    # Harmonic ratios
    sorted_indices = np.argsort(psd)[::-1]

    harmonic_ratio_1 = (
        psd[sorted_indices[1]] / (psd[sorted_indices[0]] + 1e-8)
        if len(sorted_indices) > 1 else 0
    )

    harmonic_ratio_2 = (
        psd[sorted_indices[2]] / (psd[sorted_indices[0]] + 1e-8)
        if len(sorted_indices) > 2 else 0
    )

    # Envelope spectrum features
    analytic_signal = hilbert(signal)
    envelope = np.abs(analytic_signal)

    env_freqs, env_psd = welch(
        envelope,
        fs=fs,
        nperseg=min(1024, len(envelope))
    )

    env_peak_freq = env_freqs[np.argmax(env_psd)] if len(env_freqs) > 0 else 0
    env_peak_amp = np.max(env_psd) if len(env_psd) > 0 else 0

    # FFT spectrum
    fft_vals = np.abs(rfft(signal))
    fft_freqs = rfftfreq(len(signal), d=1/fs)

    shaft_freq = rpm / 60 if rpm is not None else 0

    bpfo_freq = 4.8 * shaft_freq
    bpfi_freq = 5.2 * shaft_freq

    def get_band_amplitude(freq_array, amp_array, target_freq, bandwidth=5):
        indices = np.where(
            (freq_array >= target_freq - bandwidth) &
            (freq_array <= target_freq + bandwidth)
        )[0]

        if len(indices) == 0:
            return 0

        return np.max(amp_array[indices])

    bpfo_amp = get_band_amplitude(fft_freqs, fft_vals, bpfo_freq)
    bpfo_harmonic_2_amp = get_band_amplitude(fft_freqs, fft_vals, 2 * bpfo_freq)
    bpfo_harmonic_3_amp = get_band_amplitude(fft_freqs, fft_vals, 3 * bpfo_freq)

    bpfi_amp = get_band_amplitude(fft_freqs, fft_vals, bpfi_freq)
    bpfi_harmonic_2_amp = get_band_amplitude(fft_freqs, fft_vals, 2 * bpfi_freq)
    bpfi_harmonic_3_amp = get_band_amplitude(fft_freqs, fft_vals, 3 * bpfi_freq)

    env_harmonic_1 = get_band_amplitude(env_freqs, env_psd, env_peak_freq)
    env_harmonic_2 = get_band_amplitude(env_freqs, env_psd, 2 * env_peak_freq)
    env_harmonic_3 = get_band_amplitude(env_freqs, env_psd, 3 * env_peak_freq)

    very_low_band_energy = np.sum(psd[(freqs >= 0) & (freqs < 50)])
    low_mid_band_energy = np.sum(psd[(freqs >= 50) & (freqs < 150)])
    mid_high_band_energy = np.sum(psd[(freqs >= 150) & (freqs < 500)])
    very_high_band_energy = np.sum(psd[(freqs >= 500) & (freqs < 3000)])

    # Wavelet features
    coeffs = pywt.wavedec(signal, 'db4', level=3)

    wavelet_energy_1 = np.sum(np.square(coeffs[0]))
    wavelet_energy_2 = np.sum(np.square(coeffs[1]))
    wavelet_energy_3 = np.sum(np.square(coeffs[2]))
    wavelet_energy_4 = np.sum(np.square(coeffs[3]))

    # Rolling RMS features
    window_size = max(len(signal) // 5, 1)
    window_rms = []

    for start in range(0, len(signal) - window_size, window_size):
        segment = signal[start:start + window_size]

        if len(segment) > 0:
            segment_rms = np.sqrt(np.mean(segment ** 2))
            window_rms.append(segment_rms)

    rms_std = np.std(window_rms) if len(window_rms) > 0 else 0
    rms_max = np.max(window_rms) if len(window_rms) > 0 else 0

    # Sensor fusion features
    is_ds = 1 if placement_name == 'DS' else 0
    is_fs = 1 if placement_name == 'FS' else 0

    ds_weighted_rms = rms * is_ds
    fs_weighted_rms = rms * is_fs

    ds_weighted_kurtosis = kurt * is_ds
    fs_weighted_kurtosis = kurt * is_fs

    ds_weighted_peak = peak * is_ds
    fs_weighted_peak = peak * is_fs

    placement_interaction_rms = ds_weighted_rms - fs_weighted_rms
    placement_interaction_kurtosis = ds_weighted_kurtosis - fs_weighted_kurtosis

    return {
        'mean': mean_val,
        'std': std_val,
        'rms': rms,
        'peak': peak,
        'peak_to_peak': peak_to_peak,
        'crest_factor': crest_factor,
        'kurtosis': kurt,
        'skewness': sk,
        'dominant_freq': dominant_freq,
        'max_psd': max_psd,
        'mean_psd': mean_psd,
        'top_freq_1': top_freq_1,
        'top_freq_2': top_freq_2,
        'top_freq_3': top_freq_3,
        'top_psd_1': top_psd_1,
        'top_psd_2': top_psd_2,
        'top_psd_3': top_psd_3,
        'spectral_entropy': spectral_entropy,
        'low_band_energy': low_band_energy,
        'mid_band_energy': mid_band_energy,
        'high_band_energy': high_band_energy,
        'harmonic_ratio_1': harmonic_ratio_1,
        'harmonic_ratio_2': harmonic_ratio_2,
        'envelope_peak_freq': env_peak_freq,
        'envelope_peak_amp': env_peak_amp,
        'bpfo_amp': bpfo_amp,
        'bpfo_harmonic_2_amp': bpfo_harmonic_2_amp,
        'bpfo_harmonic_3_amp': bpfo_harmonic_3_amp,
        'bpfi_amp': bpfi_amp,
        'bpfi_harmonic_2_amp': bpfi_harmonic_2_amp,
        'bpfi_harmonic_3_amp': bpfi_harmonic_3_amp,
        'env_harmonic_1': env_harmonic_1,
        'env_harmonic_2': env_harmonic_2,
        'env_harmonic_3': env_harmonic_3,
        'very_low_band_energy': very_low_band_energy,
        'low_mid_band_energy': low_mid_band_energy,
        'mid_high_band_energy': mid_high_band_energy,
        'very_high_band_energy': very_high_band_energy,
        'wavelet_energy_1': wavelet_energy_1,
        'wavelet_energy_2': wavelet_energy_2,
        'wavelet_energy_3': wavelet_energy_3,
        'wavelet_energy_4': wavelet_energy_4,
        'window_rms_std': rms_std,
        'window_rms_max': rms_max,
        'is_ds': is_ds,
        'is_fs': is_fs,
        'ds_weighted_rms': ds_weighted_rms,
        'fs_weighted_rms': fs_weighted_rms,
        'ds_weighted_kurtosis': ds_weighted_kurtosis,
        'fs_weighted_kurtosis': fs_weighted_kurtosis,
        'ds_weighted_peak': ds_weighted_peak,
        'fs_weighted_peak': fs_weighted_peak,
        'placement_interaction_rms': placement_interaction_rms,
        'placement_interaction_kurtosis': placement_interaction_kurtosis,
        'sampling_rate': fs,
        'rpm': rpm if rpm is not None else 0
    }

In [588]:
# Cell 3: Load .mat File Function

def load_mat_file(file_path):
    mat = sio.loadmat(file_path, squeeze_me=True, struct_as_record=False)

    asset_description = str(mat.get('assetDescription', 'Unknown'))

    all_rows = []
    placements = ['DS', 'FS', 'upper', 'lower']

    for placement in placements:
        if placement not in mat:
            continue

        placement_data = mat[placement]

        try:
            raw_data = placement_data.rawData
            sampling_rates = placement_data.samplingRate
            rpms = placement_data.RPM
            labels = placement_data.label
        except:
            continue

        if not isinstance(raw_data, (list, np.ndarray)):
            raw_data = [raw_data]
            sampling_rates = [sampling_rates]
            rpms = [rpms]
            labels = [labels]

        for i in range(len(raw_data)):
            signal = raw_data[i]
            fs = sampling_rates[i] if i < len(sampling_rates) else sampling_rates[0]
            rpm = rpms[i] if i < len(rpms) else 0
            label = labels[i] if i < len(labels) else -1

            if label == -1:
                continue

            features = extract_signal_features(signal, fs, rpm,
    placement_name=placement)

            if features is None:
                continue

            features['assetDescription'] = asset_description
            features['placement'] = placement
            features['label'] = int(label)

            all_rows.append(features)

    return pd.DataFrame(all_rows)

In [589]:
# Cell 4: Load Only train.mat Files From Folders 1 to 10

base_folder = '/kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset'

all_train_data = []

for folder_num in range(1, 11):
    folder_path = os.path.join(base_folder, str(folder_num))

    train_file = os.path.join(folder_path, 'train.mat')

    if os.path.exists(train_file):
        print(f'Loading: {train_file}')

        try:
            df_train = load_mat_file(train_file)
            df_train['folder_num'] = folder_num
            all_train_data.append(df_train)

            print(f'Shape: {df_train.shape}')

        except Exception as e:
            print(f'Error in {train_file}: {e}')

train_df = pd.concat(all_train_data, ignore_index=True)

print('\nFinal Train Shape:', train_df.shape)
print(train_df.head())

Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/1/train.mat
Shape: (244, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/2/train.mat
Shape: (248, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/3/train.mat
Shape: (236, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/4/train.mat
Shape: (122, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/5/train.mat
Shape: (79, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/6/train.mat
Shape: (317, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/7/train.mat
Shape: (0, 1)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/8/train.mat
Shape: (490, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/9/train.mat
Shape: (986, 

In [590]:
# Cell 5: Keep Only Healthy Samples (label = 0)

normal_df = train_df[train_df['label'] == 0].copy()

print('Normal Samples Shape:', normal_df.shape)
print(normal_df['assetDescription'].value_counts())

Normal Samples Shape: (2839, 60)
assetDescription
Pump        1008
Agitator     986
Roller       480
Engine       365
Name: count, dtype: int64


In [591]:
# Cell 6: Encode Categorical Features

asset_encoder = LabelEncoder()
placement_encoder = LabelEncoder()

normal_df['asset_encoded'] = asset_encoder.fit_transform(normal_df['assetDescription'])
normal_df['placement_encoded'] = placement_encoder.fit_transform(normal_df['placement'])

In [592]:
# Updated Cell 7: Add Sensor Fusion Features

feature_columns = [
    'mean', 'std', 'rms', 'peak', 'peak_to_peak',
    'crest_factor', 'kurtosis', 'skewness',
    'dominant_freq', 'max_psd', 'mean_psd',
    'top_freq_1', 'top_freq_2', 'top_freq_3',
    'top_psd_1', 'top_psd_2', 'top_psd_3',
    'spectral_entropy',
    'low_band_energy',
    'mid_band_energy',
    'high_band_energy',
    'harmonic_ratio_1',
    'harmonic_ratio_2',
    'envelope_peak_freq',
    'envelope_peak_amp',
    'bpfo_amp',
    'bpfo_harmonic_2_amp',
    'bpfo_harmonic_3_amp',
    'bpfi_amp',
    'bpfi_harmonic_2_amp',
    'bpfi_harmonic_3_amp',
    'env_harmonic_1',
    'env_harmonic_2',
    'env_harmonic_3',
    'very_low_band_energy',
    'low_mid_band_energy',
    'mid_high_band_energy',
    'very_high_band_energy',
    'wavelet_energy_1',
    'wavelet_energy_2',
    'wavelet_energy_3',
    'wavelet_energy_4',
    'window_rms_std',
    'window_rms_max',
    'is_ds',
    'is_fs',
    'ds_weighted_rms',
    'fs_weighted_rms',
    'ds_weighted_kurtosis',
    'fs_weighted_kurtosis',
    'ds_weighted_peak',
    'fs_weighted_peak',
    'placement_interaction_rms',
    'placement_interaction_kurtosis',
    'sampling_rate',
    'rpm',
    'asset_encoded',
    'placement_encoded'
]

X_normal = normal_df[feature_columns]

In [593]:
# Updated Cell: Build Normal Training Features

X_normal_full = normal_df.reindex(
    columns=feature_columns,
    fill_value=0
)

X_normal_scaled = scaler.transform(X_normal_full)

print('X_normal_scaled shape:', X_normal_scaled.shape)
print('X_train_scaled shape:', X_train_scaled.shape)
print('X_test_scaled shape:', X_test_scaled.shape)

X_normal_scaled shape: (2839, 58)
X_train_scaled shape: (2839, 58)
X_test_scaled shape: (2558, 58)


In [594]:
# Replace Existing VAE Model Cell With This

import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    Dropout,
    Add,
    Reshape,
    Flatten
)
from tensorflow.keras.models import Model

input_dim = X_normal_scaled.shape[1]

print("Transformer Input Dimension:", input_dim)

inputs = Input(shape=(input_dim,))

x = Dense(128, activation='relu')(inputs)
x = Dense(64, activation='relu')(x)

# Make sure reshape works for your feature count
reshape_dim = 8
x = Dense(reshape_dim * reshape_dim, activation='relu')(x)

x_seq = Reshape((reshape_dim, reshape_dim))(x)

attention_output = MultiHeadAttention(
    num_heads=4,
    key_dim=8
)(x_seq, x_seq)

x_seq = Add()([x_seq, attention_output])
x_seq = LayerNormalization()(x_seq)

ffn = Dense(32, activation='relu')(x_seq)
ffn = Dense(reshape_dim)(ffn)

x_seq = Add()([x_seq, ffn])
x_seq = LayerNormalization()(x_seq)

x = Flatten()(x_seq)

latent = Dense(32, activation='relu', name='latent_layer')(x)

x = Dense(64, activation='relu')(latent)
x = Dense(128, activation='relu')(x)

outputs = Dense(input_dim, activation='linear')(x)

transformer_autoencoder = Model(inputs, outputs)

transformer_autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

transformer_autoencoder.summary()

Transformer Input Dimension: 58


Model: "functional_29"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_20      │ (None, 58)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_173 (Dense)   │ (None, 128)       │      7,552 │ input_layer_20[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_174 (Dense)   │ (None, 64)        │      8,256 │ dense_173[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_175 (Dense)   │ (None, 64)        │      4,160 │ dense_174[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_18          │ (None, 8, 8)      │          0 │ dense_175[0][0]   │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 8, 8)      │      1,128 │ reshape_18[0][0], │
│ (MultiHeadAttentio… │                   │            │ reshape_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_52 (Add)        │ (None, 8, 8)      │          0 │ reshape_18[0][0], │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 8, 8)      │         16 │ add_52[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_176 (Dense)   │ (None, 8, 32)     │        288 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_177 (Dense)   │ (None, 8, 8)      │        264 │ dense_176[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_53 (Add)        │ (None, 8, 8)      │          0 │ layer_normalizat… │
│                     │                   │            │ dense_177[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 8, 8)      │         16 │ add_53[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_18          │ (None, 64)        │          0 │ layer_normalizat… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latent_layer        │ (None, 32)        │      2,080 │ flatten_18[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_178 (Dense)   │ (None, 64)        │      2,112 │ latent_layer[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_179 (Dense)   │ (None, 128)       │      8,320 │ dense_178[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_180 (Dense)   │ (None, 58)        │      7,482 │ dense_179[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 41,674 (162.79 KB)

 Trainable params: 41,674 (162.79 KB)

 Non-trainable params: 0 (0.00 B)

In [595]:
# Replace Existing VAE Training Cell With This

history = transformer_autoencoder.fit(
    X_normal_scaled,
    X_normal_scaled,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=1
)

Epoch 1/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 9s 51ms/step - loss: 0.7126 - val_loss: 1.0553
Epoch 2/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1394 - val_loss: 0.9520
Epoch 3/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1538 - val_loss: 0.8789
Epoch 4/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0977 - val_loss: 0.8567
Epoch 5/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0679 - val_loss: 0.8223
Epoch 6/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0729 - val_loss: 0.9129
Epoch 7/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0622 - val_loss: 0.9553
Epoch 8/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0753 - val_loss: 0.9151
Epoch 9/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0593 - val_loss: 1.0569
Epoch 10/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0448 - val_loss: 1.0032
Epoch 11/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0502 - val_loss: 0.9043
Epoch 12/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.

In [596]:
# Replace Existing Train Reconstruction Cell With This

reconstructed_train = transformer_autoencoder.predict(
    X_train_scaled,
    verbose=0
)

train_reconstruction_error = np.mean(
    np.square(X_train_scaled - reconstructed_train),
    axis=1
)

print(train_reconstruction_error.shape)

(2839,)


In [597]:
# Replace Existing Threshold Cell With This

threshold = np.percentile(train_reconstruction_error, 80)

print('Threshold:', threshold)

Threshold: 0.16529526500851763


In [598]:
# Cell 13: Check How Many Normal Samples Are Marked As Anomalies

normal_predictions = reconstruction_error > threshold

print('Normal Samples Predicted As Anomaly:', normal_predictions.sum())
print('Normal Samples Predicted As Normal:', len(normal_predictions) - normal_predictions.sum())

Normal Samples Predicted As Anomaly: 2637
Normal Samples Predicted As Normal: 202


In [599]:
# Cell 14: Load All test.mat Files From Folders 1 to 11

all_test_data = []

for folder_num in range(1, 12):
    folder_path = os.path.join(base_folder, str(folder_num))

    test_file = os.path.join(folder_path, 'test.mat')

    if os.path.exists(test_file):
        print(f'Loading: {test_file}')

        try:
            df_test = load_mat_file(test_file)
            df_test['folder_num'] = folder_num
            all_test_data.append(df_test)

            print(f'Shape: {df_test.shape}')

        except Exception as e:
            print(f'Error in {test_file}: {e}')

test_df = pd.concat(all_test_data, ignore_index=True)

print('\nFinal Test Shape:', test_df.shape)
print(test_df.head())

Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/1/test.mat
Shape: (243, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/2/test.mat
Shape: (390, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/3/test.mat
Shape: (212, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/4/test.mat
Shape: (104, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/5/test.mat
Shape: (147, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/6/test.mat
Shape: (389, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/7/test.mat
Shape: (0, 1)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/8/test.mat
Shape: (485, 60)
Loading: /kaggle/input/datasets/sachin099x/sca-bearing-dataset/SCA bearing dataset/9/test.mat
Shape: (412, 60)
Load

In [600]:
# Cell 15: Encode Test Data

test_df['asset_encoded'] = asset_encoder.transform(test_df['assetDescription'])
test_df['placement_encoded'] = placement_encoder.transform(test_df['placement'])

X_test_full = test_df[feature_columns]
X_test_scaled = scaler.transform(X_test_full)

In [601]:
# Replace Existing Test Reconstruction Cell With This

reconstructed_test = transformer_autoencoder.predict(
    X_test_scaled,
    verbose=0
)

test_reconstruction_error = np.mean(
    np.square(X_test_scaled - reconstructed_test),
    axis=1
)

test_df['reconstruction_error'] = test_reconstruction_error

test_df['predicted_anomaly'] = (
    test_df['reconstruction_error'] > threshold
).astype(int)

print(test_df['predicted_anomaly'].value_counts())

predicted_anomaly
1    1284
0    1274
Name: count, dtype: int64


In [602]:
# New Cell: Extract Latent Features From Transformer

latent_model = Model(
    inputs=transformer_autoencoder.input,
    outputs=transformer_autoencoder.get_layer('latent_layer').output
)

latent_features = latent_model.predict(
    X_train_scaled,
    verbose=0
)

latent_feature_names = [
    f'latent_{i}' for i in range(latent_features.shape[1])
]

latent_df = pd.DataFrame(
    latent_features,
    columns=latent_feature_names
)

print(latent_df.shape)
print(latent_df.head())

(2839, 32)
   latent_0  latent_1  latent_2  latent_3  latent_4  latent_5  latent_6  \
0       0.0       0.0  0.388274       0.0       0.0  1.846387  0.768626   
1       0.0       0.0  0.317300       0.0       0.0  1.866259  1.161667   
2       0.0       0.0  0.800240       0.0       0.0  2.179909  1.013086   
3       0.0       0.0  0.372110       0.0       0.0  1.640231  0.565519   
4       0.0       0.0  0.475007       0.0       0.0  1.938138  0.634784   

   latent_7  latent_8  latent_9  ...  latent_22  latent_23  latent_24  \
0  1.211152       0.0       0.0  ...   1.597618   1.102400   2.214273   
1  1.220334       0.0       0.0  ...   1.299159   0.835821   2.273876   
2  1.098835       0.0       0.0  ...   1.486780   0.375007   1.867165   
3  1.538000       0.0       0.0  ...   1.641783   0.412035   2.077502   
4  1.622037       0.0       0.0  ...   1.901138   0.606203   2.038055   

   latent_25  latent_26  latent_27  latent_28  latent_29  latent_30  latent_31  
0   0.504579      

In [603]:
# New Cell: Extract Test Latent Features

test_latent_features = latent_model.predict(
    X_test_scaled,
    verbose=0
)

test_latent_df = pd.DataFrame(
    test_latent_features,
    columns=latent_feature_names
)

print(test_latent_df.shape)
print(test_latent_df.head())

(2558, 32)
   latent_0  latent_1  latent_2  latent_3  latent_4  latent_5  latent_6  \
0       0.0       0.0  1.393228  0.000000       0.0  3.942492  2.419297   
1       0.0       0.0  1.352821  0.000000       0.0  3.909526  2.190529   
2       0.0       0.0  1.499867  0.000000       0.0  3.889976  2.385949   
3       0.0       0.0  1.353752  0.000000       0.0  3.933561  2.330863   
4       0.0       0.0  1.374316  0.389023       0.0  4.286654  2.462523   

   latent_7  latent_8  latent_9  ...  latent_22  latent_23  latent_24  \
0  1.892687       0.0  0.797191  ...   2.638977   0.512260   2.353396   
1  2.107798       0.0  0.898380  ...   2.909330   0.449851   2.450213   
2  2.299225       0.0  0.620194  ...   2.771326   0.700597   2.530207   
3  1.983953       0.0  0.928525  ...   2.857964   0.659203   2.494252   
4  1.717455       0.0  1.407555  ...   3.401267   0.705538   2.327040   

   latent_25  latent_26  latent_27  latent_28  latent_29  latent_30  latent_31  
0   0.484143      

In [604]:
# Cell 17: Compare Anomaly Detection With True Labels

# True anomaly:
# 0 -> normal
# 1 -> anomaly (labels 1,2,3 OR folder 11 external case)

test_df['true_anomaly'] = np.where(
    (test_df['label'] == 0) & (test_df['folder_num'] != 11),
    0,
    1
)

from sklearn.metrics import classification_report

print(classification_report(
    test_df['true_anomaly'],
    test_df['predicted_anomaly']
))

              precision    recall  f1-score   support

           0       0.70      0.53      0.60      1698
           1       0.38      0.56      0.45       860

    accuracy                           0.54      2558
   macro avg       0.54      0.55      0.53      2558
weighted avg       0.59      0.54      0.55      2558



In [605]:
# Cell 18: See Predicted Anomaly Counts

print(test_df['predicted_anomaly'].value_counts())

print('\nPredicted Label Meaning:')
print('0 = Normal')
print('1 = Anomaly')

predicted_anomaly
1    1284
0    1274
Name: count, dtype: int64

Predicted Label Meaning:
0 = Normal
1 = Anomaly


In [606]:
# Cell 19: See Reconstruction Error Distribution By True Label

print(
    test_df.groupby('label')['reconstruction_error']
    .agg(['count', 'mean', 'min', 'max'])
)

       count         mean        min           max
label                                             
0.0     1736  3338.548499   0.001399  2.618873e+06
1.0      346     0.513968   0.009280  1.035472e+01
2.0       23   315.714934  14.300974  1.297349e+03
3.0      453     5.572134   0.018782  4.846403e+02


In [607]:
# Cell 20: Keep Only Predicted Anomalies

predicted_anomaly_df = test_df[
    test_df['predicted_anomaly'] == 1
].copy()

print('Predicted Anomalies Shape:', predicted_anomaly_df.shape)

print(predicted_anomaly_df[['label', 'folder_num']].head())

Predicted Anomalies Shape: (1284, 65)
   label  folder_num
0    1.0           1
1    1.0           1
2    1.0           1
3    1.0           1
4    1.0           1


In [608]:
# Cell 21: Create Bearing vs External Labels

# Folder 11 -> external anomaly
# Folders 1 to 10 with labels 1,2,3 -> bearing anomaly

predicted_anomaly_df['anomaly_type'] = np.where(
    predicted_anomaly_df['folder_num'] == 11,
    0,
    1
)

print(predicted_anomaly_df['anomaly_type'].value_counts())

print('\nLabel Meaning:')
print('0 = External / Non-Bearing')
print('1 = Bearing Fault')

anomaly_type
1    1246
0      38
Name: count, dtype: int64

Label Meaning:
0 = External / Non-Bearing
1 = Bearing Fault


In [609]:
from sklearn.model_selection import train_test_split

In [610]:
# Updated Cell 22: XGBoost For Bearing vs External Classifier

X_anomaly = predicted_anomaly_df[feature_columns]
y_anomaly = predicted_anomaly_df['anomaly_type']

X_train_anom, X_test_anom, y_train_anom, y_test_anom = train_test_split(
    X_anomaly,
    y_anomaly,
    test_size=0.2,
    random_state=42,
    stratify=y_anomaly
)

anomaly_classifier = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

anomaly_classifier.fit(X_train_anom, y_train_anom)

y_pred_anom = anomaly_classifier.predict(X_test_anom)

In [611]:
# Cell 23: Bearing vs External Classification Report + Confusion Matrix

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_test_anom,
    y_pred_anom
))

cm_anom = confusion_matrix(
    y_test_anom,
    y_pred_anom,
    labels=[0, 1]
)

print('\nConfusion Matrix:')
print(cm_anom)

print('\nLabel Meaning:')
print('0 = External / Non-Bearing')
print('1 = Bearing Fault')

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00       249

    accuracy                           1.00       257
   macro avg       1.00      1.00      1.00       257
weighted avg       1.00      1.00      1.00       257


Confusion Matrix:
[[  8   0]
 [  0 249]]

Label Meaning:
0 = External / Non-Bearing
1 = Bearing Fault


In [613]:
# Stage 3 Fault Classifier - Excluding Folders 8, 9, and 11
exclude_folders = [8, 9, 11]

# Pool all fault data (Labels 1, 2, 3)
bearing_fault_pool = pd.concat([
    train_df[train_df['label'].isin([1, 2, 3])],
    test_df[test_df['label'].isin([1, 2, 3])]
], ignore_index=True)

# Strictly exclude the target test folders from the training pool
bearing_fault_train_pool = bearing_fault_pool[~bearing_fault_pool['folder_num'].isin(exclude_folders)]

print(f"Training Stage 3 on Folders: {sorted(bearing_fault_train_pool['folder_num'].unique())}")
print(f"Excluded for Testing: {exclude_folders}")

X_fault = bearing_fault_train_pool[feature_columns]
y_fault = bearing_fault_train_pool['label']

# XGBoost expects labels from 0 (1->0, 2->1, 3->2)
y_fault_xgb = y_fault.map({1: 0, 2: 1, 3: 2})

# Split the training data
X_train_fault, X_test_fault, y_train_fault, y_test_fault = train_test_split(
    X_fault,
    y_fault_xgb,
    test_size=0.30,
    random_state=42,
    stratify=y_fault_xgb
)

# Apply SMOTE to balance the faults
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_fault_smote, y_train_fault_smote = smote.fit_resample(X_train_fault, y_train_fault)

# Compute sample weights
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_fault_smote)

# Model configuration
fault_classifier = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss'
)

fault_classifier.fit(
    X_train_fault_smote,
    y_train_fault_smote,
    sample_weight=sample_weights
)

# --- FIX: Compare using .values to avoid Index Alignment Errors ---
y_pred_fault_raw = fault_classifier.predict(X_test_fault)
reverse_label_map = {0: 1, 1: 2, 2: 3}

y_test_fault_mapped = pd.Series(y_test_fault).map(reverse_label_map)
y_pred_fault_mapped = pd.Series(y_pred_fault_raw).map(reverse_label_map)

# Use .values or .to_numpy() to compare the underlying data
accuracy = (y_test_fault_mapped.values == y_pred_fault_mapped.values).mean()
print(f"Stage 3 Validation Accuracy on Pool: {accuracy:.2%}")

Training Stage 3 on Folders: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(10)]
Excluded for Testing: [8, 9, 11]
Stage 3 Validation Accuracy on Pool: 100.00%


In [616]:
def final_pipeline_predict(sample_row):
    # Convert row to DataFrame for model compatibility
    sample_df = pd.DataFrame([sample_row])
    sample_features = sample_df[feature_columns]
    
    # 1. Stage 1: Autoencoder reconstruction error
    sample_scaled = scaler.transform(sample_features)
    reconstructed = transformer_autoencoder.predict(sample_scaled, verbose=0)
    
    recon_error = np.mean(
        np.square(sample_scaled - reconstructed),
        axis=1
    )[0]

    # 2. Stage 2: Get XGBoost Anomaly Probability
    # anomaly_classifier was trained to distinguish Healthy vs Anomaly
    # [Class 0 = Normal, Class 1 = Anomaly]
    anomaly_probs = anomaly_classifier.predict_proba(sample_features)[0]
    xgb_anomaly_score = anomaly_probs[1] # Probability of being an anomaly

    # 3. HYBRID GATE:
    # If the Autoencoder error is high OR XGBoost is confident it's an anomaly
    # We use a lower threshold (0.4) for XGBoost to be sensitive to Folders 8 & 9
    if (recon_error > threshold) or (xgb_anomaly_score > 0.4):
        
        # Stage 3: Fault classification with confidence filtering
        fault_probs = fault_classifier.predict_proba(sample_features)[0]
        
        confidence = np.max(fault_probs)
        predicted_class_xgb = np.argmax(fault_probs)

        # Confidence filter to avoid misclassifying noise as a fault
        if confidence < 0.60:
            return 0

        reverse_label_map = {0: 1, 1: 2, 2: 3}
        fault_type = reverse_label_map[predicted_class_xgb]
        return int(fault_type)

    # If both models agree it's normal, return 0
    return 0

In [617]:
# Cell 25: Bearing Fault Type Report
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_test_fault_mapped, 
    y_pred_fault_mapped,
    zero_division=0
))

cm_fault = confusion_matrix(
    y_test_fault_mapped,
    y_pred_fault_mapped,
    labels=[1, 2, 3]
)

print('\nConfusion Matrix:')
print(cm_fault)

              precision    recall  f1-score   support

           1       1.00      1.00      1.00       104
           2       1.00      1.00      1.00         7
           3       1.00      1.00      1.00        70

    accuracy                           1.00       181
   macro avg       1.00      1.00      1.00       181
weighted avg       1.00      1.00      1.00       181


Confusion Matrix:
[[104   0   0]
 [  0   7   0]
 [  0   0  70]]


In [618]:
# Cell 26: Fault Prediction Counts

pred_counts = pd.Series(y_pred_fault).value_counts().sort_index()

label_map = {
    1: 'Inner Ring Fault',
    2: 'Ball Fault',
    3: 'Outer Ring Fault'
}

print('\nPredicted Label Counts:')

for label in [1, 2, 3]:
    count = pred_counts.get(label, 0)
    print(f'Class {label} ({label_map[label]}): {count} samples')


Predicted Label Counts:
Class 1 (Inner Ring Fault): 104 samples
Class 2 (Ball Fault): 7 samples
Class 3 (Outer Ring Fault): 136 samples


In [619]:
def final_pipeline_predict(sample_row):
    sample_df = pd.DataFrame([sample_row])
    sample_features = sample_df[feature_columns]
    
    # 1. Stage 1: Autoencoder reconstruction error
    sample_scaled = scaler.transform(sample_features)
    reconstructed = transformer_autoencoder.predict(sample_scaled, verbose=0)
    
    recon_error = np.mean(
        np.square(sample_scaled - reconstructed),
        axis=1
    )[0]

    # 2. Stage 2: Get XGBoost Anomaly Probability
    anomaly_probs = anomaly_classifier.predict_proba(sample_features)[0]
    xgb_anomaly_score = anomaly_probs[1] 

    # 3. OPTIMIZED HYBRID GATE:
    # Increased XGBoost threshold from 0.4 to 0.85 to reduce False Positives
    # Also added a check for the Autoencoder error (even if below threshold, it shouldn't be zero)
    if (recon_error > threshold) or (xgb_anomaly_score > 0.85):
        
        # Stage 3: Fault classification with stricter confidence
        fault_probs = fault_classifier.predict_proba(sample_features)[0]
        
        confidence = np.max(fault_probs)
        predicted_class_xgb = np.argmax(fault_probs)

        # Increased confidence requirement to 0.80 to ensure high-quality labels
        if confidence < 0.80:
            return 0

        reverse_label_map = {0: 1, 1: 2, 2: 3}
        fault_type = reverse_label_map[predicted_class_xgb]
        return int(fault_type)

    return 0

In [620]:
# Isolate all samples from Folders 8, 9, 11
test_data_8_9_11 = test_df[test_df['folder_num'].isin([8, 9, 11])].copy()

final_predictions_isolated = []
true_labels_isolated = []

print(f"Processing {len(test_data_8_9_11)} samples from Folders 8, 9, & 11...")

for idx, row in test_data_8_9_11.iterrows():
    # Run the full Hybrid Pipeline
    predicted_label = final_pipeline_predict(row)
    
    # Ground truth mapping:
    # Folder 11 is external disturbance -> True Label 0
    if row['folder_num'] == 11:
        true_label = 0
    else:
        true_label = int(row['label'])
        
    final_predictions_isolated.append(predicted_label)
    true_labels_isolated.append(true_label)

# Final Results Reporting
from sklearn.metrics import classification_report, confusion_matrix

print("\n--- PIPELINE REPORT (UNSEEN FOLDERS 8, 9, 11) ---")
print(classification_report(
    true_labels_isolated, 
    final_predictions_isolated, 
    labels=[0, 1, 2, 3], 
    zero_division=0
))

cm_isolated = confusion_matrix(
    true_labels_isolated, 
    final_predictions_isolated, 
    labels=[0, 1, 2, 3]
)

print("\nConfusion Matrix (Rows: True, Cols: Pred):")
print(cm_isolated)

print("\nLegend:")
print("0: Normal / External Disturbance")
print("1: Inner Ring Fault")
print("2: Ball Fault")
print("3: Outer Ring Fault")

Processing 935 samples from Folders 8, 9, & 11...

--- PIPELINE REPORT (UNSEEN FOLDERS 8, 9, 11) ---
              precision    recall  f1-score   support

           0       0.97      0.64      0.77       714
           1       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         0
           3       0.91      0.93      0.92       221

    accuracy                           0.70       935
   macro avg       0.47      0.39      0.42       935
weighted avg       0.95      0.70      0.80       935


Confusion Matrix (Rows: True, Cols: Pred):
[[454 240   0  20]
 [  0   0   0   0]
 [  0   0   0   0]
 [ 16   0   0 205]]

Legend:
0: Normal / External Disturbance
1: Inner Ring Fault
2: Ball Fault
3: Outer Ring Fault
